In [1]:
import pandas as pd

df = pd.read_csv("results/results.csv")
df.head()

,trial_id,condition,outcome_type,amount,run_number,model,temperature,prompt_version,prompt_text,response_text,parsed_wager,log_wager,valid,refusal_flag,parse_error_type,timestamp,request_id
0,1,paper_loss_small,paper,-200,1,anthropic/claude-3.5-haiku,1.0,absolute,You are currently on a casino visit. So far du...,50,50.0,3.912023,True,False,NaN,2026-03-18T12:33:02.968469+00:00,gen-1773837181-P5K8wiFF21na5b9v54q0
1,2,realized_medium_loss,realized,-400,1,anthropic/claude-3.5-haiku,1.0,absolute,You visited this casino on a previous occasion...,100,100.0,4.605170,True,False,NaN,2026-03-18T12:33:28.256586+00:00,gen-1773837207-68TwBWef8uA75rlH3xAV
2,3,paper_loss_large,paper,-350,1,anthropic/claude-3.5-haiku,1.0,absolute,You are currently on a casino visit. So far du...,100,100.0,4.605170,True,False,NaN,2026-03-18T12:34:12.834339+00:00,gen-1773837251-wUIrjjnp6sxINdL0iqZY
3,4,paper_gain_large,paper,150,1,anthropic/claude-3.5-haiku,1.0,absolute,You are currently on a casino visit. So far du...,50,50.0,3.912023,True,False,NaN,2026-03-18T12:34:20.617303+00:00,gen-1773837259-It9kEZ82b6lpbEAye21G
4,5,paper_gain_small,paper,40,1,anthropic/claude-3.5-haiku,1.0,absolute,You are currently on a casino visit. So far du...,50,50.0,3.912023,True,False,NaN,2026-03-18T12:38:49.109109+00:00,gen-1773837528-vNX0e8DEdsm8RdKexncv


In [2]:
df = df[df["valid"] == True].copy()
df["parsed_wager"] = pd.to_numeric(df["parsed_wager"])
df["amount"] = pd.to_numeric(df["amount"])
df["temperature"] = pd.to_numeric(df["temperature"])

In [3]:
df["log_wager"] = pd.to_numeric(df["log_wager"], errors="coerce")

In [4]:
df["is_loss"] = (df["amount"] < 0).astype(int)

In [5]:
df["is_realized"] = (df["outcome_type"] == "realized").astype(int)

In [6]:
df["loss_realized"] = df["is_loss"] * df["is_realized"]

In [7]:
import statsmodels.api as sm

X = df[["is_loss", "is_realized", "loss_realized"]]
X = sm.add_constant(X)

y = df["parsed_wager"]   # or use "log_wager"

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     181.5
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          3.96e-115
Time:                        11:22:50   Log-Likelihood:                -67380.
No. Observations:               12138   AIC:                         1.348e+05
Df Residuals:                   12134   BIC:                         1.348e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            92.2822      1.033     89.318

In [8]:
df["abs_amount"] = df["amount"].abs()
df["log_amount"] = df["abs_amount"].apply(lambda x: np.log(x + 1))

NameError: name 'np' is not defined

In [9]:
import numpy as np

In [10]:
df["abs_amount"] = df["amount"].abs()
df["log_amount"] = df["abs_amount"].apply(lambda x: np.log(x + 1))

In [11]:
model = sm.OLS(y, X).fit(cov_type='HC3')
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.043
Method:                 Least Squares   F-statistic:                     173.2
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          5.56e-110
Time:                        11:33:48   Log-Likelihood:                -67380.
No. Observations:               12138   AIC:                         1.348e+05
Df Residuals:                   12134   BIC:                         1.348e+05
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            92.2822      0.921    100.187

In [12]:
X = df[["is_loss", "is_realized", "loss_realized", "log_amount"]]

In [13]:
df.groupby("model").apply(lambda x: ...)

model
anthropic/claude-3.5-haiku              Ellipsis
anthropic/claude-haiku-4.5              Ellipsis
deepseek/deepseek-chat-v3-0324          Ellipsis
google/gemini-3.1-flash-lite-preview    Ellipsis
gpt-4.1-mini                            Ellipsis
moonshotai/kimi-k2-thinking             Ellipsis
openai/gpt-5.4-mini                     Ellipsis
qwen/qwen3-32b                          Ellipsis
x-ai/grok-4-fast                        Ellipsis
dtype: object

In [14]:
import statsmodels.api as sm

results = {}

for model_name in df["model"].unique():
    sub = df[df["model"] == model_name]

    X = sub[["is_loss", "is_realized", "loss_realized"]]
    X = sm.add_constant(X)
    y = sub["parsed_wager"]

    model = sm.OLS(y, X).fit(cov_type="HC3")
    
    results[model_name] = model.params

# Convert to DataFrame
results_df = pd.DataFrame(results).T
results_df

,const,is_loss,is_realized,loss_realized
anthropic/claude-3.5-haiku,58.333333,4.166667,-1.333333,36.748333
anthropic/claude-haiku-4.5,46.833333,0.166667,3.166667,-0.541667
deepseek/deepseek-chat-v3-0324,176.426174,-13.144924,-24.911023,69.649823
google/gemini-3.1-flash-lite-preview,53.233333,1.716667,5.566667,38.483333
gpt-4.1-mini,93.792282,-38.302996,3.664861,36.095139
openai/gpt-5.4-mini,61.766667,-20.786667,-14.166667,36.236667
qwen/qwen3-32b,127.694301,73.930699,-2.349473,-18.361348
moonshotai/kimi-k2-thinking,46.476510,16.686755,13.223490,52.231335
x-ai/grok-4-fast,86.833333,6.166667,14.166667,23.333333


In [15]:
anthropic/claude-3.5-haiku	58.333333	4.166667	-1.333333	36.748333
anthropic/claude-haiku-4.5	46.833333	0.166667	3.166667	-0.541667
deepseek/deepseek-chat-v3-0324	176.426174	-13.144924	-24.911023	69.649823
google/gemini-3.1-flash-lite-preview	53.233333	1.716667	5.566667	38.483333
gpt-4.1-mini	93.792282	-38.302996	3.664861	36.095139
openai/gpt-5.4-mini	61.766667	-20.786667	-14.166667	36.236667
qwen/qwen3-32b	127.694301	73.930699	-2.349473	-18.361348
moonshotai/kimi-k2-thinking	46.476510	16.686755	13.223490	52.231335
x-ai/grok-4-fast	86.833333	6.166667	14.166667	23.333333


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2373896214.py, line 3)

In [16]:
plot_df = pd.DataFrame(full_results)

NameError: name 'full_results' is not defined

In [17]:
X = ["is_loss", "is_realized", "loss_realized", "log_amount"]

In [18]:
+ log_amount

NameError: name 'log_amount' is not defined

In [19]:
import pandas as pd

df = pd.read_csv("results/results.csv")

In [20]:
import numpy as np

df["abs_amount"] = df["amount"].abs()
df["log_amount"] = np.log(df["abs_amount"] + 1)

df["is_loss"] = (df["amount"] < 0).astype(int)
df["is_realized"] = (df["outcome_type"] == "realized").astype(int)
df["loss_realized"] = df["is_loss"] * df["is_realized"]

In [21]:
import statsmodels.api as sm

X = df[["is_loss", "is_realized", "loss_realized", "log_amount"]]
X = sm.add_constant(X)

y = df["parsed_wager"]

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())

ValueError: r_matrix performs f_test for using dimensions that are asymptotically non-normal

In [22]:
X.dtypes
X.nunique()
X.isna().sum()
np.isinf(X).sum()

const            0
is_loss          0
is_realized      0
loss_realized    0
log_amount       0
dtype: int64

In [23]:
y.isna().sum(), np.isinf(y).sum()

(np.int64(17), np.int64(0))

In [24]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("results/results.csv").copy()

df = df[df["valid"] == True].copy()

df["parsed_wager"] = pd.to_numeric(df["parsed_wager"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df["abs_amount"] = df["amount"].abs()
df["log_amount"] = np.log(df["abs_amount"] + 1)

df["is_loss"] = (df["amount"] < 0).astype(int)
df["is_realized"] = (df["outcome_type"] == "realized").astype(int)
df["loss_realized"] = (df["is_loss"] * df["is_realized"]).astype(int)

cols = ["parsed_wager", "is_loss", "is_realized", "loss_realized", "log_amount"]
df = df[cols].replace([np.inf, -np.inf], np.nan).dropna()

X = df[["is_loss", "is_realized", "loss_realized", "log_amount"]].astype(float)
X = sm.add_constant(X)
y = df["parsed_wager"].astype(float)

model = sm.OLS(y, X).fit(cov_type="HC3")

In [25]:
import statsmodels.api as sm

X = df[["is_loss", "is_realized", "loss_realized", "log_amount"]]
X = sm.add_constant(X)

y = df["parsed_wager"]

model = sm.OLS(y, X).fit(cov_type="HC3")
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           parsed_wager   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.053
Method:                 Least Squares   F-statistic:                     149.5
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          5.19e-125
Time:                        11:44:45   Log-Likelihood:                -67312.
No. Observations:               12138   AIC:                         1.346e+05
Df Residuals:                   12133   BIC:                         1.347e+05
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            80.6859      1.453     55.532